In [ ]:
# En Google Colab: descomentar las dos líneas siguientes
# from google.colab import drive
# drive.mount('/content/drive')


In [ ]:
# En Google Colab: descomentar y ajustar la ruta
# import os
# os.chdir('/content/drive/MyDrive/Courses/AI/masked_attention/llama_like/excercises')


In [ ]:
import sys, os
# Agrega el directorio padre (donde está src/) al path de Python
parent = os.path.abspath(os.path.join(os.getcwd(), '..'))
if parent not in sys.path:
    sys.path.insert(0, parent)


In [ ]:
"""
Taller: Internals de un LLM estilo LLaMA
=================================================================
Instrucciones:
  - Busca los bloques marcados con TODO y completa el código.
  - Cada sección tiene una celda de verificación al final.
  - No modifiques nada fuera de los bloques TODO.

Secciones:
   4. RoPE ablation — qué pasa sin codificación posicional
"""

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import matplotlib.pyplot as plt
from dataclasses import dataclass
from typing import Optional

# Importamos el modelo de referencia para comparaciones
from src.model import (
    ModelConfig, RMSNorm, SwiGLUFFN, MiniLLaMA,
    precompute_rope_freqs, apply_rope, GroupedQueryAttention
)
from src.data import get_corpus
from src.tokenizer import BPETokenizer

In [ ]:
# ===========================================================================
# SECCIÓN 4 — RoPE ablation
# ===========================================================================
# ¿Qué pasa si entrenamos sin codificación posicional?
# Compara la curva de loss de un modelo con RoPE vs uno sin RoPE.
#
# Para quitar RoPE, basta con no rotar Q y K antes del dot product.
# ===========================================================================

class AttentionNoRoPE(nn.Module):
    """GroupedQueryAttention sin rotación posicional."""

    def __init__(self, config: ModelConfig):
        super().__init__()
        assert config.n_heads % config.n_kv_heads == 0
        self.n_heads    = config.n_heads
        self.n_kv_heads = config.n_kv_heads
        self.n_rep      = config.n_heads // config.n_kv_heads
        self.head_dim   = config.d_model // config.n_heads
        D, Dh           = config.d_model, self.head_dim

        self.Wq = nn.Linear(D, config.n_heads    * Dh, bias=False)
        self.Wk = nn.Linear(D, config.n_kv_heads * Dh, bias=False)
        self.Wv = nn.Linear(D, config.n_kv_heads * Dh, bias=False)
        self.Wo = nn.Linear(config.n_heads * Dh, D,    bias=False)

    def forward(self, x, rope_freqs, mask=None):
        B, T, D = x.shape
        Dh      = self.head_dim

        q = self.Wq(x).reshape(B, T, self.n_heads,    Dh)
        k = self.Wk(x).reshape(B, T, self.n_kv_heads, Dh)
        v = self.Wv(x).reshape(B, T, self.n_kv_heads, Dh)

        # TODO 4.1 — Esta versión NO aplica RoPE a Q y K.
        # (simplemente no llamamos apply_rope — las queries y keys
        #  no llevan información posicional rotacional)

        k = k.unsqueeze(3).expand(B, T, self.n_kv_heads, self.n_rep, Dh)\
             .reshape(B, T, self.n_heads, Dh)
        v = v.unsqueeze(3).expand(B, T, self.n_kv_heads, self.n_rep, Dh)\
             .reshape(B, T, self.n_heads, Dh)

        q = q.transpose(1, 2)
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)

        scale  = math.sqrt(Dh)
        scores = torch.matmul(q, k.transpose(-2, -1)) / scale
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float("-inf"))
        attn = F.softmax(scores, dim=-1)
        out  = torch.matmul(attn, v)
        out  = out.transpose(1, 2).reshape(B, T, self.n_heads * Dh)
        return self.Wo(out)


# ---------------------------------------------------------------------------
# Dataset de ventana deslizante (mismo que train.ipynb)
# ---------------------------------------------------------------------------
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
import os

class TextDataset(Dataset):
    def __init__(self, token_ids: list, seq_len: int):
        self.seq_len = seq_len
        self.data    = torch.tensor(token_ids, dtype=torch.long)
        self.n       = max(0, len(self.data) - seq_len - 1)

    def __len__(self):
        return self.n

    def __getitem__(self, idx):
        x = self.data[idx       : idx + self.seq_len]
        y = self.data[idx + 1   : idx + self.seq_len + 1]
        return x, y


def train_quick(use_rope: bool, epochs: int = 100) -> list:
    """Entrena MiniLLaMA con o sin RoPE y retorna la curva de loss por epoch."""
    # TODO 4.2 — Implementación completa

    # ── Hiperparámetros ───────────────────────────────────────────────────
    SEQ_LEN    = 64
    BATCH_SIZE = 8
    LR         = 3e-3
    WEIGHT_DECAY = 0.01
    SEED       = 42

    torch.manual_seed(SEED)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # ── 1. Tokenizer ──────────────────────────────────────────────────────
    # Cargamos el tokenizer pre-entrenado para garantizar el mismo vocabulario
    # en ambas condiciones y hacer la comparación completamente justa.
    tok_path = os.path.join(os.path.dirname(__file__) if "__file__" in dir() else ".",
                            "..", "checkpoints", "tokenizer.json")
    # Ruta relativa al notebook (excercises/)
    tok_path = os.path.join("..", "checkpoints", "tokenizer.json")
    tokenizer = BPETokenizer()
    tokenizer.load(tok_path)

    # ── 2. Corpus y tokens ────────────────────────────────────────────────
    corpus    = get_corpus(os.path.join("..", "corpus.txt"))
    token_ids = tokenizer.encode(corpus, add_special_tokens=False)

    # ── 3. Dataset y DataLoader ───────────────────────────────────────────
    dataset    = TextDataset(token_ids, seq_len=SEQ_LEN)
    dataloader = DataLoader(dataset, batch_size=BATCH_SIZE,
                            shuffle=True, drop_last=True)

    # ── 4. Modelo ─────────────────────────────────────────────────────────
    cfg = ModelConfig(
        vocab_size  = tokenizer.vocab_size,
        d_model     = 128,
        n_heads     = 4,
        n_kv_heads  = 2,
        d_ff        = 256,
        max_seq_len = SEQ_LEN,
        dropout     = 0.0,   # sin dropout para comparación limpia
    )
    model = MiniLLaMA(cfg).to(device)

    # ── 5. Sustituir módulo de atención si use_rope=False ─────────────────
    if not use_rope:
        model.layer.attn = AttentionNoRoPE(cfg).to(device)
        # Re-inicializar pesos con la misma semilla para comparación justa
        torch.manual_seed(SEED)
        for m in model.modules():
            if isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, mean=0.0, std=0.02)

    # ── 6. Optimizador con cosine scheduler ───────────────────────────────
    optimizer   = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    total_steps = epochs * max(len(dataloader), 1)
    scheduler   = CosineAnnealingLR(optimizer, T_max=total_steps, eta_min=1e-5)

    # ── 7. Bucle de entrenamiento ─────────────────────────────────────────
    model.train()
    losses = []
    for epoch in range(1, epochs + 1):
        epoch_loss = 0.0
        n_batches  = 0
        for x, y in dataloader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            _, loss = model(x, targets=y)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()
            epoch_loss += loss.item()
            n_batches  += 1
        losses.append(epoch_loss / max(n_batches, 1))

    return losses


# ── Verificación 4 ──────────────────────────────────────────────────────────
def verify_section4():
    print("=" * 55)
    print("VERIFICACIÓN 4 — RoPE ablation  (entrena ~100 epochs)")
    print("=" * 55)

    losses_rope    = train_quick(use_rope=True,  epochs=100)
    losses_no_rope = train_quick(use_rope=False, epochs=100)

    final_rope    = losses_rope[-1]
    final_no_rope = losses_no_rope[-1]
    print(f"  Loss final con RoPE    : {final_rope:.4f}")
    print(f"  Loss final sin RoPE    : {final_no_rope:.4f}")
    print(f"  RoPE mejora el loss    : {final_rope < final_no_rope}")

    plt.figure(figsize=(7, 4))
    plt.plot(losses_rope,    label="con RoPE",    linewidth=1.5)
    plt.plot(losses_no_rope, label="sin RoPE",    linewidth=1.5, linestyle="--")
    plt.xlabel("epoch")
    plt.ylabel("loss")
    plt.title("RoPE ablation — curva de entrenamiento")
    plt.legend()
    plt.tight_layout()
    plt.savefig("rope_ablation.png", dpi=130)
    print("  Plot guardado en rope_ablation.png")

    if final_rope < final_no_rope:
        print("\n  ✓ Sección 4 correcta")
    else:
        print("\n  ~ Resultado inesperado — revisa train_quick()")


In [ ]:
verify_section4()